# Create AI video with Magic Hour MCP

Connect CrewAI 1.15.22 to Magic Hour's hosted MCP server. Magic Hour exposes image, video, and audio generation tools. This quickstart discovers tools without spending credits, then optionally runs one text-to-video task.

Prerequisites:
- Python 3.10–3.13
- [Magic Hour API key](https://magichour.ai/developer)
- `OPENAI_API_KEY` for the optional crew run


In [ ]:
%pip install -q "crewai==1.15.22" "crewai-tools[mcp]==1.15.22"


In [ ]:
import os

from crewai import Agent, Crew, Process, Task
from crewai_tools import MCPServerAdapter

assert os.environ.get("MAGIC_HOUR_API_KEY"), "Set MAGIC_HOUR_API_KEY before continuing."

server_params = {
    "url": "https://mcp.magichour.ai/",
    "headers": {"Authorization": f"Bearer {os.environ['MAGIC_HOUR_API_KEY']}"},
    "transport": "streamable-http",
}


## Discover tools

This connects and lists tools only. It does not start a generation job or spend Magic Hour credits.


In [ ]:
with MCPServerAdapter(server_params, connect_timeout=60) as tools:
    print(f"Discovered {len(tools)} Magic Hour tools")
    print("\n".join(tool.name for tool in tools))


## Create one video

The next cell starts a credit-consuming render. It creates one project, waits on that same project ID, and asks the agent to return the exact signed download URL without trimming query parameters.


In [ ]:
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running the crew."

with MCPServerAdapter(server_params, connect_timeout=60) as tools:
    producer = Agent(
        role="AI video producer",
        goal="Create one polished video and return its exact download URL",
        backstory=(
            "You produce short videos with Magic Hour. Create each project once, wait for the "
            "same project ID to finish, and preserve signed download URLs exactly as returned."
        ),
        tools=tools,
        verbose=True,
    )
    task = Task(
        description=(
            "Create a five-second cinematic video of bioluminescent jellyfish drifting through "
            "deep ocean water. Wait for completion and return the exact download URL."
        ),
        expected_output="The completed project ID and exact signed video download URL.",
        agent=producer,
    )
    result = Crew(agents=[producer], tasks=[task], process=Process.sequential).kickoff()
    print(result)
